# Tables Secondaires

In [5]:
import pandas as pd
import numpy as np

app_train = pd.read_csv("../donnees/traitees/app_train_prepare.csv")
app_test = pd.read_csv("../donnees/traitees/app_test_prepare.csv")

print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train : (307503, 226)
app_test : (48744, 225)


In [6]:
# J'ajoute la fonction du on hot encoding
def encodage_one_hot(df, nan_as_category=True):
    colonnes_avant = list(df.columns)
    colonnes_categorielles = df.select_dtypes(include=["object", "str"]).columns.tolist()
    df = pd.get_dummies(df, columns=colonnes_categorielles, dummy_na=nan_as_category)
    nouvelles_colonnes = [c for c in df.columns if c not in colonnes_avant]
    return df, nouvelles_colonnes

## Table bureau / bureau_balance
Agrégation générique (mean/min/max/sum) au niveau SK_ID_CURR, inspirée du kernel jsaguiar

In [7]:
bureau = pd.read_csv("../donnees/brutes/bureau.csv")
bureau_balance = pd.read_csv("../donnees/brutes/bureau_balance.csv")

print("bureau :", bureau.shape)
print("bureau_balance :", bureau_balance.shape)

bureau : (1716428, 17)
bureau_balance : (27299925, 3)


In [8]:
# Encodage des catégorielles
bureau_balance, colonnes_cat_bb = encodage_one_hot(bureau_balance, nan_as_category=True)
bureau, colonnes_cat_bureau = encodage_one_hot(bureau, nan_as_category=True)

# Agrégation de bureau_balance au niveau SK_ID_BUREAU (1ligne=1crédit)
colonnes_numeriques_bb = [col for col in bureau_balance.columns if col not in ["SK_ID_BUREAU"] + colonnes_cat_bb]

agg_numerique = bureau_balance.groupby("SK_ID_BUREAU")[colonnes_numeriques_bb].agg(["min", "max", "mean", "size"])
agg_numerique.columns = ["_".join(col).upper() for col in agg_numerique.columns]

agg_categoriel = bureau_balance.groupby("SK_ID_BUREAU")[colonnes_cat_bb].mean()
agg_categoriel.columns = [col + "_MEAN" for col in agg_categoriel.columns]

agg_bureau_balance = agg_numerique.join(agg_categoriel)

# Fusion avec bureau (au niveau du crédit)
bureau = bureau.join(agg_bureau_balance, how="left", on="SK_ID_BUREAU")
bureau = bureau.drop(columns=["SK_ID_BUREAU"])

# Agrégation finale au niveau SK_ID_CURR (1ligne=1client)
agg_bureau = bureau.groupby("SK_ID_CURR").agg(["min", "max", "mean", "sum"])
agg_bureau.columns = ["BUREAU_" + "_".join(col).upper() for col in agg_bureau.columns]
agg_bureau = agg_bureau.reset_index()

print("Dimensions bureau agrégé :", agg_bureau.shape)
agg_bureau.head()

C:\Users\yavas\AppData\Local\Temp\ipykernel_54748\2885703240.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_bureau = agg_bureau.reset_index()


Dimensions bureau agrégé : (305811, 205)


,SK_ID_CURR,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_SUM,BUREAU_CREDIT_DAY_OVERDUE_MIN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MIN,...,BUREAU_STATUS_C_MEAN_MEAN,BUREAU_STATUS_C_MEAN_SUM,BUREAU_STATUS_X_MEAN_MIN,BUREAU_STATUS_X_MEAN_MAX,BUREAU_STATUS_X_MEAN_MEAN,BUREAU_STATUS_X_MEAN_SUM,BUREAU_STATUS_NAN_MEAN_MIN,BUREAU_STATUS_NAN_MEAN_MAX,BUREAU_STATUS_NAN_MEAN_MEAN,BUREAU_STATUS_NAN_MEAN_SUM
0,100001,-1572,-49,-735.000000,-5145,0,0,0.0,0,-1329.0,...,0.441240,3.088683,0.0,0.500000,0.214590,1.502129,0.0,0.0,0.0,0.0
1,100002,-1437,-103,-874.000000,-6992,0,0,0.0,0,-1072.0,...,0.175426,1.403409,0.0,0.500000,0.161932,1.295455,0.0,0.0,0.0,0.0
2,100003,-2586,-606,-1400.750000,-5603,0,0,0.0,0,-2434.0,...,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.0
3,100004,-1326,-408,-867.000000,-1734,0,0,0.0,0,-595.0,...,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.0
4,100005,-373,-62,-190.666667,-572,0,0,0.0,0,-128.0,...,0.128205,0.384615,0.0,0.333333,0.136752,0.410256,0.0,0.0,0.0,0.0


### Fusion de bureau_agg avec application_train / application_test

In [9]:
print("Dimensions avant fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train = app_train.join(agg_bureau.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")
app_test = app_test.join(agg_bureau.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")

print("Dimensions après fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions avant fusion :
app_train : (307503, 226)
app_test : (48744, 225)
Dimensions après fusion :
app_train : (307503, 430)
app_test : (48744, 429)


In [10]:
import gc

del bureau, bureau_balance, agg_bureau, agg_bureau_balance, agg_numerique, agg_categoriel
gc.collect()

1720

## Table previous_application
Même anomalie que DAYS_EMPLOYED (365243 = code, pas une vraie valeur) présente sur 5 colonnes de dates. On s'inspire de jsaguiar : traitement de l'anomalie, création d'un ratio métier (APP_CREDIT_PERC), puis agrégation par client — avec un focus supplémentaire sur les demandes Approuvées vs Refusées séparément.

In [11]:
previous_application = pd.read_csv("../donnees/brutes/previous_application.csv")
print("Dimensions initiales :", previous_application.shape)

# Encodage des catégorielles
previous_application, colonnes_cat_prev = encodage_one_hot(previous_application, nan_as_category=True)

# Anomalie 365243 -> NaN sur les 5 colonnes de dates concernées
colonnes_dates_anomalie = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
for col in colonnes_dates_anomalie:
    previous_application[col] = previous_application[col].replace(365243, np.nan)

# Feature métier : pourcentage demandé vs accordé
previous_application["APP_CREDIT_PERC"] = previous_application["AMT_APPLICATION"] / previous_application["AMT_CREDIT"]

print("Dimensions après préparation :", previous_application.shape)

Dimensions initiales : (1670214, 37)
Dimensions après préparation : (1670214, 181)


In [12]:
# Séparation numérique / catégoriel pour l'agrégation
colonnes_numeriques_prev = [col for col in previous_application.columns if col not in ["SK_ID_CURR", "SK_ID_PREV"] + colonnes_cat_prev]

# Agrégation globale (toutes les demandes précédentes)
agg_num = previous_application.groupby("SK_ID_CURR")[colonnes_numeriques_prev].agg(["min", "max", "mean"])
agg_num.columns = ["_".join(col).upper() for col in agg_num.columns]

agg_cat = previous_application.groupby("SK_ID_CURR")[colonnes_cat_prev].mean()
agg_cat.columns = [col + "_MEAN" for col in agg_cat.columns]

agg_previous = agg_num.join(agg_cat)
agg_previous.columns = ["PREV_" + col for col in agg_previous.columns]

# Agrégation séparée : demandes Approuvées
approuvees = previous_application[previous_application["NAME_CONTRACT_STATUS_Approved"] == 1]
agg_approuvees = approuvees.groupby("SK_ID_CURR")[colonnes_numeriques_prev].agg(["min", "max", "mean"])
agg_approuvees.columns = ["APPROVED_" + "_".join(col).upper() for col in agg_approuvees.columns]

# Agrégation séparée : demandes Refusées
refusees = previous_application[previous_application["NAME_CONTRACT_STATUS_Refused"] == 1]
agg_refusees = refusees.groupby("SK_ID_CURR")[colonnes_numeriques_prev].agg(["min", "max", "mean"])
agg_refusees.columns = ["REFUSED_" + "_".join(col).upper() for col in agg_refusees.columns]

# Fusion des 3 agrégations
agg_previous = agg_previous.join(agg_approuvees, how="left").join(agg_refusees, how="left")
agg_previous = agg_previous.reset_index()

print("Dimensions previous_application agrégé :", agg_previous.shape)
agg_previous.head()

Dimensions previous_application agrégé : (338857, 340)


,SK_ID_CURR,PREV_AMT_ANNUITY_MIN,PREV_AMT_ANNUITY_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_APPLICATION_MIN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_CREDIT_MIN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_MEAN,...,REFUSED_DAYS_LAST_DUE_MEAN,REFUSED_DAYS_TERMINATION_MIN,REFUSED_DAYS_TERMINATION_MAX,REFUSED_DAYS_TERMINATION_MEAN,REFUSED_NFLAG_INSURED_ON_APPROVAL_MIN,REFUSED_NFLAG_INSURED_ON_APPROVAL_MAX,REFUSED_NFLAG_INSURED_ON_APPROVAL_MEAN,REFUSED_APP_CREDIT_PERC_MIN,REFUSED_APP_CREDIT_PERC_MAX,REFUSED_APP_CREDIT_PERC_MEAN
0,100001,3951.000,3951.000,3951.000,24835.5,24835.5,24835.50,23787.0,23787.0,23787.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,9251.775,9251.775,9251.775,179055.0,179055.0,179055.00,179055.0,179055.0,179055.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100003,6737.310,98356.995,56553.990,68809.5,900000.0,435436.50,68053.5,1035882.0,484191.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,5357.250,5357.250,5357.250,24282.0,24282.0,24282.00,20106.0,20106.0,20106.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,4813.200,4813.200,4813.200,0.0,44617.5,22308.75,0.0,40153.5,20076.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Fusion de agg_previous avec app_train/app_test

In [13]:
print("Dimensions avant fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train = app_train.join(agg_previous.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")
app_test = app_test.join(agg_previous.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")

print("Dimensions après fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions avant fusion :
app_train : (307503, 430)
app_test : (48744, 429)
Dimensions après fusion :
app_train : (307503, 769)
app_test : (48744, 768)


In [14]:
del previous_application, agg_previous, agg_num, agg_cat, approuvees, refusees, agg_approuvees, agg_refusees
gc.collect()

0

## Table POS_CASH_balance
Historique mensuel des crédits POS/cash précédents. Agrégation directe au niveau client, plus un comptage du nombre de lignes (= nombre de mois d'historique cumulés).

In [15]:
pos_cash = pd.read_csv("../donnees/brutes/POS_CASH_balance.csv")
print("Dimensions initiales :", pos_cash.shape)

# Encodage des catégorielles
pos_cash, colonnes_cat_pos = encodage_one_hot(pos_cash, nan_as_category=True)

# Séparation numérique / catégoriel
colonnes_numeriques_pos = [col for col in pos_cash.columns if col not in ["SK_ID_CURR", "SK_ID_PREV"] + colonnes_cat_pos]

agg_num = pos_cash.groupby("SK_ID_CURR")[colonnes_numeriques_pos].agg(["max", "mean"])
agg_num.columns = ["_".join(col).upper() for col in agg_num.columns]

agg_cat = pos_cash.groupby("SK_ID_CURR")[colonnes_cat_pos].mean()
agg_cat.columns = [col + "_MEAN" for col in agg_cat.columns]

agg_pos = agg_num.join(agg_cat)
agg_pos.columns = ["POS_" + col for col in agg_pos.columns]

# Comptage du nombre de lignes (mois d'historique) par client
agg_pos["POS_COUNT"] = pos_cash.groupby("SK_ID_CURR").size()

agg_pos = agg_pos.reset_index()

print("Dimensions POS_CASH agrégé :", agg_pos.shape)
agg_pos.head()

Dimensions initiales : (10001358, 8)
Dimensions POS_CASH agrégé : (337252, 22)


,SK_ID_CURR,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_DEF_MAX,...,POS_NAME_CONTRACT_STATUS_Amortized debt_MEAN,POS_NAME_CONTRACT_STATUS_Approved_MEAN,POS_NAME_CONTRACT_STATUS_Canceled_MEAN,POS_NAME_CONTRACT_STATUS_Completed_MEAN,POS_NAME_CONTRACT_STATUS_Demand_MEAN,POS_NAME_CONTRACT_STATUS_Returned to the store_MEAN,POS_NAME_CONTRACT_STATUS_Signed_MEAN,POS_NAME_CONTRACT_STATUS_XNA_MEAN,POS_NAME_CONTRACT_STATUS_nan_MEAN,POS_COUNT
0,100001,-53,-72.555556,4.0,4.000000,4.0,1.444444,7,0.777778,7,...,0.0,0.0,0.0,0.222222,0.0,0.0,0.000000,0.0,0.0,9
1,100002,-1,-10.000000,24.0,24.000000,24.0,15.000000,0,0.000000,0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,19
2,100003,-18,-43.785714,12.0,10.107143,12.0,5.785714,0,0.000000,0,...,0.0,0.0,0.0,0.071429,0.0,0.0,0.000000,0.0,0.0,28
3,100004,-24,-25.500000,4.0,3.750000,4.0,2.250000,0,0.000000,0,...,0.0,0.0,0.0,0.250000,0.0,0.0,0.000000,0.0,0.0,4
4,100005,-15,-20.000000,12.0,11.700000,12.0,7.200000,0,0.000000,0,...,0.0,0.0,0.0,0.090909,0.0,0.0,0.090909,0.0,0.0,11


### Fusion de pos_cash avec app_train/app_test

In [16]:
print("Dimensions avant fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train = app_train.join(agg_pos.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")
app_test = app_test.join(agg_pos.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")

print("Dimensions après fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions avant fusion :
app_train : (307503, 769)
app_test : (48744, 768)
Dimensions après fusion :
app_train : (307503, 790)
app_test : (48744, 789)


In [17]:
del pos_cash, agg_pos, agg_num, agg_cat
gc.collect()

0

## Table installments_payments
Historique des échéances remboursées. Features métier : écart entre montant dû/payé, retard/avance de paiement (DPD/DBD), puis agrégation au niveau client.

In [18]:
installments = pd.read_csv("../donnees/brutes/installments_payments.csv")
print("Dimensions initiales :", installments.shape)

# Encodage des catégorielles 
installments, colonnes_cat_ins = encodage_one_hot(installments, nan_as_category=True)

# Features métier : écart et pourcentage payé vs dû
installments["PAYMENT_PERC"] = installments["AMT_PAYMENT"] / installments["AMT_INSTALMENT"]
installments["PAYMENT_DIFF"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]

# DPD (Days Past Due : nombre de jours de retard de paiement) et DBD (Days Before Due:nombre de jours d'avance de paiement), jamais négatifs
installments["DPD"] = (installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]).clip(lower=0)
installments["DBD"] = (installments["DAYS_INSTALMENT"] - installments["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

print("Dimensions après préparation :", installments.shape)

Dimensions initiales : (13605401, 8)
Dimensions après préparation : (13605401, 12)


In [19]:
# Agrégation
# Séparation numérique / catégoriel
colonnes_numeriques_ins = [col for col in installments.columns if col not in ["SK_ID_CURR", "SK_ID_PREV"] + colonnes_cat_ins]

agg_installments = installments.groupby("SK_ID_CURR")[colonnes_numeriques_ins].agg(["max", "mean", "sum"])
agg_installments.columns = ["INSTAL_" + "_".join(col).upper() for col in agg_installments.columns]

# Comptage du nombre d'échéances par client
agg_installments["INSTAL_COUNT"] = installments.groupby("SK_ID_CURR").size()

agg_installments = agg_installments.reset_index()

print("Dimensions installments agrégé :", agg_installments.shape)
agg_installments.head()

Dimensions installments agrégé : (339587, 32)


,SK_ID_CURR,INSTAL_NUM_INSTALMENT_VERSION_MAX,INSTAL_NUM_INSTALMENT_VERSION_MEAN,INSTAL_NUM_INSTALMENT_VERSION_SUM,INSTAL_NUM_INSTALMENT_NUMBER_MAX,INSTAL_NUM_INSTALMENT_NUMBER_MEAN,INSTAL_NUM_INSTALMENT_NUMBER_SUM,INSTAL_DAYS_INSTALMENT_MAX,INSTAL_DAYS_INSTALMENT_MEAN,INSTAL_DAYS_INSTALMENT_SUM,...,INSTAL_PAYMENT_DIFF_MAX,INSTAL_PAYMENT_DIFF_MEAN,INSTAL_PAYMENT_DIFF_SUM,INSTAL_DPD_MAX,INSTAL_DPD_MEAN,INSTAL_DPD_SUM,INSTAL_DBD_MAX,INSTAL_DBD_MEAN,INSTAL_DBD_SUM,INSTAL_COUNT
0,100001,2.0,1.142857,8.0,4,2.714286,19,-1619.0,-2187.714286,-15314.0,...,0.0,0.0,0.0,11.0,1.571429,11.0,36.0,8.857143,62.0,7
1,100002,2.0,1.052632,20.0,19,10.000000,190,-25.0,-295.000000,-5605.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,31.0,20.421053,388.0,19
2,100003,2.0,1.040000,26.0,12,5.080000,127,-536.0,-1378.160000,-34454.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,14.0,7.160000,179.0,25
3,100004,2.0,1.333333,4.0,3,2.000000,6,-724.0,-754.000000,-2262.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,11.0,7.666667,23.0,3
4,100005,2.0,1.111111,10.0,9,5.000000,45,-466.0,-586.000000,-5274.0,...,0.0,0.0,0.0,1.0,0.111111,1.0,37.0,23.666667,213.0,9


### Fusion installments_payments avec app_train/app_test

In [20]:
print("Dimensions avant fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train = app_train.join(agg_installments.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")
app_test = app_test.join(agg_installments.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")

print("Dimensions après fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions avant fusion :
app_train : (307503, 790)
app_test : (48744, 789)
Dimensions après fusion :
app_train : (307503, 821)
app_test : (48744, 820)


In [21]:
del installments, agg_installments
gc.collect()

0

## Table credit_card_balance
Historique mensuel des cartes de crédit précédentes. Agrégation directe (min/max/mean/sum) sur toutes les colonnes numériques

In [22]:
credit_card = pd.read_csv("../donnees/brutes/credit_card_balance.csv")
print("Dimensions initiales :", credit_card.shape)

# Encodage des catégorielles
credit_card, colonnes_cat_cc = encodage_one_hot(credit_card, nan_as_category=True)
credit_card = credit_card.drop(columns=["SK_ID_PREV"])

# Séparation numérique / catégoriel
colonnes_numeriques_cc = [col for col in credit_card.columns if col not in ["SK_ID_CURR"] + colonnes_cat_cc]

agg_num = credit_card.groupby("SK_ID_CURR")[colonnes_numeriques_cc].agg(["min", "max", "mean", "sum"])
agg_num.columns = ["_".join(col).upper() for col in agg_num.columns]

agg_cat = credit_card.groupby("SK_ID_CURR")[colonnes_cat_cc].mean()
agg_cat.columns = [col + "_MEAN" for col in agg_cat.columns]

agg_cc = agg_num.join(agg_cat)
agg_cc.columns = ["CC_" + col for col in agg_cc.columns]

# Comptage du nombre de lignes (mois) par client
agg_cc["CC_COUNT"] = credit_card.groupby("SK_ID_CURR").size()

agg_cc = agg_cc.reset_index()

print("Dimensions credit_card agrégé :", agg_cc.shape)
agg_cc.head()

Dimensions initiales : (3840312, 23)
Dimensions credit_card agrégé : (103558, 90)


,SK_ID_CURR,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_MEAN,CC_MONTHS_BALANCE_SUM,CC_AMT_BALANCE_MIN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MIN,...,CC_SK_DPD_DEF_SUM,CC_NAME_CONTRACT_STATUS_Active_MEAN,CC_NAME_CONTRACT_STATUS_Approved_MEAN,CC_NAME_CONTRACT_STATUS_Completed_MEAN,CC_NAME_CONTRACT_STATUS_Demand_MEAN,CC_NAME_CONTRACT_STATUS_Refused_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_COUNT
0,100006,-6,-1,-3.5,-21,0.0,0.00,0.000000,0.000,270000,...,0,1.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,6
1,100011,-75,-2,-38.5,-2849,0.0,189000.00,54482.111149,4031676.225,90000,...,0,1.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,74
2,100013,-96,-1,-48.5,-4656,0.0,161420.22,18159.919219,1743352.245,45000,...,1,1.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,96
3,100021,-18,-2,-10.0,-170,0.0,0.00,0.000000,0.000,675000,...,0,0.411765,0.0,0.588235,0.0,0.0,0.0,0.0,0.0,17
4,100023,-11,-4,-7.5,-60,0.0,0.00,0.000000,0.000,45000,...,0,1.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,8


### Fusion credit_card_balanced avec app_train/app_test

In [23]:
print("Dimensions avant fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

app_train = app_train.join(agg_cc.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")
app_test = app_test.join(agg_cc.set_index("SK_ID_CURR"), how="left", on="SK_ID_CURR")

print("Dimensions après fusion :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions avant fusion :
app_train : (307503, 821)
app_test : (48744, 820)
Dimensions après fusion :
app_train : (307503, 910)
app_test : (48744, 909)


In [24]:
del credit_card, agg_cc, agg_num, agg_cat
gc.collect()

0

## Gestion des valeurs manquantes issues des fusions

In [26]:
for nom, df in [("app_train", app_train), ("app_test", app_test)]:
    valeurs_manquantes = df.isnull().sum().sort_values(ascending=False)
    valeurs_manquantes = valeurs_manquantes[valeurs_manquantes > 0]
    print(f"{nom} : {len(valeurs_manquantes)} colonnes concernées par des NaN")

print()
# Détail pour app_train
valeurs_manquantes = app_train.isnull().sum().sort_values(ascending=False)
valeurs_manquantes = valeurs_manquantes[valeurs_manquantes > 0]
pourcentage_manquantes = (valeurs_manquantes / len(app_train) * 100).round(2)

tableau_manquantes = pd.DataFrame({
    "nb_manquantes": valeurs_manquantes,
    "pct_manquantes": pourcentage_manquantes
})
tableau_manquantes.head(30)

app_train : 684 colonnes concernées par des NaN
app_test : 684 colonnes concernées par des NaN



,nb_manquantes,pct_manquantes
REFUSED_RATE_INTEREST_PRIVILEGED_MIN,307503,100.0
REFUSED_RATE_INTEREST_PRIMARY_MEAN,307503,100.0
REFUSED_DAYS_LAST_DUE_1ST_VERSION_MEAN,307503,100.0
REFUSED_DAYS_TERMINATION_MEAN,307503,100.0
REFUSED_DAYS_TERMINATION_MIN,307503,100.0
REFUSED_DAYS_TERMINATION_MAX,307503,100.0
REFUSED_NFLAG_INSURED_ON_APPROVAL_MAX,307503,100.0
REFUSED_NFLAG_INSURED_ON_APPROVAL_MEAN,307503,100.0
REFUSED_NFLAG_INSURED_ON_APPROVAL_MIN,307503,100.0
REFUSED_DAYS_LAST_DUE_MEAN,307503,100.0


In [27]:
seuil_suppression = 0.80  # 80% de NaN

taux_manquants = app_train.isnull().mean().sort_values(ascending=False)
colonnes_a_supprimer_nan = taux_manquants[taux_manquants > seuil_suppression]

print(f"Nombre de colonnes à plus de {seuil_suppression*100}% de NaN : {len(colonnes_a_supprimer_nan)}")
print()
colonnes_a_supprimer_nan

Nombre de colonnes à plus de 80.0% de NaN : 69



REFUSED_RATE_INTEREST_PRIVILEGED_MIN      1.000000
REFUSED_RATE_INTEREST_PRIMARY_MEAN        1.000000
REFUSED_DAYS_LAST_DUE_1ST_VERSION_MEAN    1.000000
REFUSED_DAYS_TERMINATION_MEAN             1.000000
REFUSED_DAYS_TERMINATION_MIN              1.000000
                                            ...   
CC_AMT_DRAWINGS_POS_CURRENT_MIN           0.801176
CC_CNT_DRAWINGS_ATM_CURRENT_MIN           0.801176
CC_CNT_DRAWINGS_POS_CURRENT_MIN           0.801176
CC_CNT_DRAWINGS_POS_CURRENT_MEAN          0.801176
CC_CNT_DRAWINGS_POS_CURRENT_MAX           0.801176
Length: 69, dtype: float64

 Suppression des colonnes à plus de 80% de NaN (peu d'information fiable).

In [30]:
# Suppression des colonnes à plus de 80% de NaN
app_train = app_train.drop(columns=colonnes_a_supprimer_nan.index)
app_test = app_test.drop(columns=[col for col in colonnes_a_supprimer_nan.index if col in app_test.columns])

print("Dimensions après suppression :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions après suppression :
app_train : (307503, 841)
app_test : (48744, 840)


Imputation par 0 pour les colonnes comptages (COUNT, SIZE, SUM) : une absence d'historique vaut factuellement 0, pas une valeur inconnue.   
Imputation par médiane pour le reste (MIN, MAX, MEAN) : on ne peut pas déduire une vraie valeur, la médiane reste la meilleure estimation.

In [32]:
# Imputation par 0 : colonnes de comptage
colonnes_comptage = [col for col in app_train.columns if any(mot in col for mot in ["COUNT", "SIZE", "SUM"])]
print(f"\n{len(colonnes_comptage)} colonnes de comptage imputées à 0")

for col in colonnes_comptage:
    app_train[col] = app_train[col].fillna(0)
    if col in app_test.columns:
        app_test[col] = app_test[col].fillna(0)

# Imputation par médiane : tout le reste
colonnes_a_imputer_mediane = [col for col in app_train.columns if col not in ["SK_ID_CURR", "TARGET"] + colonnes_comptage]

for col in colonnes_a_imputer_mediane:
    mediane = app_train[col].median()
    valeur_remplacement = mediane if not pd.isna(mediane) else 0
    app_train[col] = app_train[col].fillna(valeur_remplacement)
    if col in app_test.columns:
        app_test[col] = app_test[col].fillna(valeur_remplacement)

print("\nValeurs manquantes après imputation complète :")
print("app_train :", app_train.isnull().sum().sum())
print("app_test :", app_test.isnull().sum().sum())


102 colonnes de comptage imputées à 0

Valeurs manquantes après imputation complète :
app_train : 0
app_test : 0


## Sauvegarde du dataset final

In [33]:
app_train.to_csv("../donnees/traitees/dataset_final_train.csv", index=False)
app_test.to_csv("../donnees/traitees/dataset_final_test.csv", index=False)

print("Dimensions finales :")
print("app_train :", app_train.shape)
print("app_test :", app_test.shape)

Dimensions finales :
app_train : (307503, 841)
app_test : (48744, 840)


## Vérification doublons

In [35]:
# Vérification qu'il n'y a pas de doublons de SK_ID_CURR (colonne portant l'information "unique")
print("\nDoublons SK_ID_CURR dans app_train :", app_train["SK_ID_CURR"].duplicated().sum())
print("Doublons SK_ID_CURR dans app_test :", app_test["SK_ID_CURR"].duplicated().sum())


Doublons SK_ID_CURR dans app_train : 0
Doublons SK_ID_CURR dans app_test : 0
